In [ ]:
!pip install langchain langchian-core langchain-community pydantic duckduckgo-search

In [ ]:
# built-in Tool - DuckDuckGo Search
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun() # these are runnable, these have invoke method
results = search_tool.invoke('ipl news')

print(results)

In [ ]:
results = search_tool.invoke('top news in India today')
print(results)

In [ ]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

In [ ]:
!pip install langchain-experimental # install for shell tool below

In [ ]:
# built-in Tool - Shell Tool
from langchain_community.tools import ShellTool

shell_tool = ShellTool()
results = shell_tool.invoke('whoami')
results

In [ ]:
results = shell_tool.invoke('ls')
results

In [ ]:
# there are lots of tools - read LangChain documentation
# while calling tools, we send tool schema to llm (present in built-in tools), we can create schema in our custom tools

In [ ]:
# custom tools
# there are multiple ways to create own tool, here we are seeing simplest one

In [ ]:
# METHOD - 1
from langchain_core.tools import tool

# docstring is highly recommended to add in tools bcz in future llm will refer this docstring to understand working of the tool
# step 1 - create a function
def multiply(a, b):
    """Multiply two numbers"""
    return a*b

# type hinting is also recommended for understanding llm what type of data it will get in return from this tool
# step 2 - add type hints
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    retuen a*b

# step 3 - add tool decorator
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    retuen a*b


In [ ]:
result = multiply.invoke({'a': 3, 'b':5}) # as tool is a runnable, it has runnable
print(result)

In [ ]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

In [ ]:
# METHOD - 2: Using Structured Tool
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The first number to add")

def multiply_func(a: int, b: int) -> int:
    return a*b

multiply_tool = StructuredTool.from_function(
    func = multiply_func,
    name = "multiply",
    description = "Multiply two numbers",
    args_schema = MultiplyInput
)

result = multiply.invoke({'a': 3, 'b':5}) # as tool is a runnable, it has runnable
print(result)

print(multiply.name)
print(multiply.description)
print(multiply.args)


In [ ]:
# METHOD - 3: Uing BaseTool Class (# name of run method should be exactly `_run`)
# can create async version in this method, can't crete async method in previous methods
from langchain.tools import BaseTool
from typing import Type

# arg schema using pydantic
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The first number to add")

class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"
    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a*b

multiply_tool = MultiplyTool()

result = multiply_tool.invoke({'a': 3, 'b': 5})

### ToolKit

In [ ]:
from langchain_core.tools import tool

# custom tools
@tool
def add(a: int, b:int) -> int:
    """Add two numbers"""
    return a+b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a*b

class MathToolkit:
    def get_tools(self):
        return [add, multiply]

toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


## Tool Calling

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = ""

In [ ]:
!pip install -q langchain-openai langchain-core requests

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [ ]:
# tool create
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    retuen a*b

In [ ]:

print(multiply.invoke({'a': 3, 'b': 5}))

In [ ]:
multiply.name

In [ ]:
multiply.description

In [ ]:
multiply.args

In [ ]:
# tool binding
llm = ChatOpenAI()

# add ohther tools by putting comma if there are more than one tool
llm_with_tools = llm.bind_toolss([multiply]) # few llms are able to d tool binding (able to work with toosls)

In [ ]:
llm_with_tools.invoke('Hi how are you?') # no tool calling

In [ ]:
llm_with_tools.invoke('Can you multiply 3 with 10?') # tool calling happening # here llm can only suggest tools required, will not call tool, if llm will execute tool calling that will be risky

In [ ]:
llm_with_tools.invoke('Can you multiply 3 with 10?').tool_calls

In [ ]:
llm_with_tools.invoke('Can you multiply 3 with 10?').tool_calls[0]

In [ ]:
# tool execution
result = llm_with_tools.invoke('Can you multiply 3 with 10?')
result.tool_calls[0]['args']

In [ ]:
multiply.invoke(result.tool_calls[0]['args'])

In [ ]:
multiply.invoke(result.tool_calls[0]) # will get ToolMessage

In [ ]:
# maintain conversation history
query = 'Can you multiply 3 with 10?'
messages = [query]
print(messages)
result = llm_with_tools.invoke(messages)
print(result)
messages.append(result)
print(messages)
tool_result = multiply.invoke(result.tool_calls[0])
messages.append(result)
print(messages)
print(multiply.invoke(result.tool_calls[0]))
print(multiply.invoke(result.tool_calls[0]).content)

In [ ]:
# try with new query
query = 'Can you multiply 3 with 1000?'
messages = [query]
print(messages)
result = llm_with_tools.invoke(messages)
print(result)
messages.append(result)
print(messages)
tool_result = multiply.invoke(result.tool_calls[0])
messages.append(result)
print(messages)
print(multiply.invoke(result.tool_calls[0]))
print(multiply.invoke(result.tool_calls[0]).content)

### Currency Conversion Tool

In [ ]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between base currency and a target currency
    """
    url = f"" # write exchange rate url
    response = requests.get(url)
    return response.json

In [ ]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})

In [ ]:
@tool
def convert(base_currency_value: int, conversion_rate: float) -> float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    return base_currency_value*conversion_rate

In [ ]:
convert.invoke({'base_currency_value': 10, 'conversion_rate': 85.16})

In [ ]:
# tool binding
llm = ChatOpenAI()
llm_with_tools = llm.bind_toolss([get_conversion_factor, convert])

messages = HumanMessages('What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?')
ai_message = llm_with_tools.invoke(messagesmessages)
ai_message.tool_calls

In [ ]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between base currency and a target currency
    """
    url = f"" # write exchange rate url
    response = requests.get(url)
    return response.json

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    return base_currency_value*conversion_rate

# tool binding
llm = ChatOpenAI()
llm_with_tools = llm.bind_toolss([get_conversion_factor, convert])

messages = HumanMessages('What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?')
ai_message = llm_with_tools.invoke(messages)
ai_message.tool_calls
messages.append(ai_message)

In [ ]:
for tool_call in ai_message.tool_calls:
    print(tool_call)
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        print(tool_message1)
        # fetch this conversion rate
        print(tool_message1.content)
        # print(tool_message1.content['conversion_rate'])
        print(json.loads(tool_message1.content)['conversion_rate'])
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        # append this tool message to messages list
        messages.append(tool_message1)
    # execute the 2nd tool using the conversion rate from tool 1
    if tool_call['name'] == 'convert':
        # fetch the current argument
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [ ]:
print(messages) # three types of Messages will be here

In [ ]:
llm_with_tools.invoke(messages).content

In [ ]:
# try with INR to USD